# 00 · Two ways to scrape Tiki: the rendered page vs the JSON API

> Run locally: requires Google Chrome (Selenium starts it in headless mode). Notebooks 01 and 02 do not need it.

The first scraping method I learned during my internship was to read the page a person sees: open it in a browser with **Selenium**, then parse the HTML with **BeautifulSoup**. Looking at DevTools → Network later showed that tiki.vn loads its search results from a **JSON API**, and that API is what the pipeline (`crawler/tiki.py`) uses. This notebook runs both methods on the same keyword and the same first page of results, and compares them.

Both runs are polite: one page load in Chrome, one plain page download and one API request. `robots.txt` allows `/search?q=` and `/api/v2/products` for normal keywords (see notebook 01).

In [1]:
import sys
import time
from pathlib import Path
from urllib.parse import quote

import pandas as pd
import requests
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from crawler import tiki, tiki_html

pd.set_option("display.max_colwidth", 50)
KEYWORD = "chuột gaming"

## 1. Selenium + BeautifulSoup

`crawler/tiki_html.py` opens `tiki.vn/search?q=…` in headless Chrome, waits until the first product card appears (`WebDriverWait`), scrolls three times with a 2-second pause, then reads every card with BeautifulSoup: name (`h3`), price (`.price-discount__price`), units sold (`.quantity`), rating (the width of the star bar), link and image. Sponsored cards (marked "AD") are skipped.

In [2]:
start = time.time()
html = tiki_html.fetch_rendered_html(KEYWORD)    # headless Chrome: load, wait, scroll
from_html = tiki_html.parse_search_html(html)    # BeautifulSoup: read the product cards
html_seconds = time.time() - start

print(f"{len(from_html)} products in {html_seconds:.1f} s")
from_html[["name", "price", "sold", "rating", "brand"]].head()

40 products in 14.0 s


,name,price,sold,rating,brand
0,Chuột Gaming Logitech G102 Gen 2 Lightsync - H...,471900,4757,4.8,LOGITECH
1,Chuột Gaming Có Dây Newmen Game G10+ Hàng Chín...,199000,18,5.0,NEWMEN
2,Chuột Gaming E-DRA EM6102w - Hàng Chính Hãng,243000,6,NaN,E-DRA
3,Chuột cơ gaming led RGB 12800DPI MK500 mechani...,645000,22,5.0,TECH77
4,Chuột LED RGB 7200DPI Gaming Macro Mouse C8 ch...,235000,14,5.0,TECH77


### Why open a browser at all?

Because the product cards are not in the HTML the server sends: the page's JavaScript draws them after loading. A plain `requests.get()` of the same page finds none.

In [3]:
raw_html = requests.get(tiki_html.SEARCH_PAGE + quote(KEYWORD), headers=tiki.HEADERS, timeout=15).text
cards_without_browser = BeautifulSoup(raw_html, "html.parser").select(tiki_html.PRODUCT_CARD)
print("product cards in the page sent by the server:", len(cards_without_browser))
print("product cards after Chrome ran the page:     ", len(BeautifulSoup(html, "html.parser").select(tiki_html.PRODUCT_CARD)),
      "(sponsored cards included)")

product cards in the page sent by the server: 0
product cards after Chrome ran the page:      58 (sponsored cards included)


## 2. The JSON API

`crawler/tiki.py` sends one request to `tiki.vn/api/v2/products`, the endpoint the search page itself calls (found in DevTools → Network). The answer is JSON, so every field already has a name: there is no HTML to parse.

In [4]:
start = time.time()
data = tiki.get_page(KEYWORD, page=1)                              # one request, JSON answer
from_api = pd.DataFrame([tiki.parse_item(item) for item in data["data"]])
api_seconds = time.time() - start

print(f"{len(from_api)} products in {api_seconds:.1f} s")
print("fields per product in the raw JSON:", len(data["data"][0]))
from_api[["name", "price", "sold", "rating", "brand"]].head()

40 products in 0.4 s
fields per product in the raw JSON: 46


,name,price,sold,rating,brand
0,Chuột Gaming Logitech G102 Gen 2 Lightsync - H...,471900,4757,4.8,Logitech
1,Chuột Gaming Có Dây Newmen Game G10+ Hàng Chín...,199000,18,5.0,Newmen
2,Chuột Gaming E-DRA EM6102w - Hàng Chính Hãng,243000,6,0.0,E-DRA
3,Chuột cơ gaming led RGB 12800DPI MK500 mechani...,645000,22,5.0,Tech77
4,Chuột LED RGB 7200DPI Gaming Macro Mouse C8 ch...,235000,14,5.0,Tech77


## 3. Comparison

Same keyword, same first page. Both methods return 13 columns (the schema of `parse_item`); the question is how many of them actually contain data, and whether the shared values agree.

In [5]:
def filled_columns(df):
    """Columns that have a value for at least one product."""
    return [column for column in df.columns if df[column].notna().any()]


comparison = pd.DataFrame({
    "Selenium + BeautifulSoup": [round(html_seconds, 1), len(from_html), len(filled_columns(from_html))],
    "JSON API": [round(api_seconds, 1), len(from_api), len(filled_columns(from_api))],
}, index=["time (s)", "products", "columns with data (of 13)"])

both = from_html.merge(from_api, on="product_id", suffixes=("_html", "_api"))
rated = both[both["rating_html"].notna()]
print("columns missing in the HTML:", sorted(set(filled_columns(from_api)) - set(filled_columns(from_html))))
print("products found by both:", len(both))
print("same price:", int((both["price_html"] == both["price_api"]).sum()), "of", len(both))
print("same units sold:", int((both["sold_html"] == both["sold_api"]).sum()), "of", len(both))
print("same rating (cards that show stars):", int((rated["rating_html"] == rated["rating_api"].round(1)).sum()), "of", len(rated))
comparison

columns missing in the HTML: ['category', 'original_price', 'review_count', 'seller']
products found by both: 40
same price: 40 of 40
same units sold: 40 of 40
same rating (cards that show stars): 14 of 14


,Selenium + BeautifulSoup,JSON API
time (s),14.0,0.4
products,40.0,40.0
columns with data (of 13),9.0,13.0


| | Selenium + BeautifulSoup | JSON API |
|---|---|---|
| Time for one page | over 10 s: start Chrome, load the page, 3 × 2 s scroll pauses | under 1 s: one request |
| Products | 40 (the sponsored "AD" cards are skipped) | 40, the same products |
| Columns with data | 9 of 13: no category, seller, original price or review count; brand only in capitals ("LOGITECH") | 13 of 13, picked from 46 raw fields |
| Values | same prices, units sold and ratings as the API | |
| Rating | read from the width of a star bar (96 % → 4.8) | a number, `rating_average` |
| When the website changes | breaks if Tiki renames a class like `price-discount__price` or changes the card layout | keeps working while the JSON field names stay the same; they do not depend on the page design |
| Needs | Chrome + Selenium | only `requests` |

The time row changes from run to run; the printed outputs above are the numbers of this run.

## 4. Conclusion

**Why the pipeline uses the API.** It returns the same products with the same values, many times faster, and with the fields the analysis needs most: the category (every statistic is computed per category), the review count (deal score) and the original price (discounts). It does not depend on how the page looks, and it is lighter for Tiki's servers: one small JSON answer instead of a full page with scripts and images.

**When Selenium is still the right tool:**

- the data only exists in the rendered page (no API call to find in DevTools, or the API needs something the page computes first);
- you need to check what a shopper actually sees: layout, badges, sponsored cards;
- the page needs an action before it shows the data, such as clicking "load more" or choosing a filter.

Here it stays a learning and comparison tool: no login, no CAPTCHA solving and no attempt to hide that the browser is automated (Chrome reports itself as `HeadlessChrome`).